# 번호판 인식 전체 파이프라인 (Detection + OCR)

YOLO로 번호판 검출 → EasyOCR로 문자 인식까지 전체 과정을 Colab에서 실행합니다.

## 파이프라인 구조

```
입력 이미지 → [Step 1] YOLO 검출 → [Step 2+3] EasyOCR → 번호판 텍스트
```

## 실행 전 확인
1. **런타임 → 런타임 유형 변경 → GPU (T4)** 선택
2. 순서대로 셀 실행

---

## 0. GPU 확인

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU 선택")

## 1. 패키지 설치

In [ ]:
# YOLO + EasyOCR 설치
!pip install ultralytics easyocr scikit-learn -q
print("✅ 패키지 설치 완료!")

---

# Part A: Step 1 - YOLO 번호판 검출 모델 학습

이미 학습된 모델이 있으면 **Part B**로 건너뛰세요.

---

## 2. Kaggle 데이터셋 다운로드

In [ ]:
# Kaggle API 키 업로드
from google.colab import files
print("kaggle.json 파일을 업로드하세요:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API 설정 완료!")

In [ ]:
# 데이터 다운로드 (andrewmvd/car-plate-detection - 433장 + XML 라벨)
!kaggle datasets download -d andrewmvd/car-plate-detection
!unzip -q car-plate-detection.zip -d data/
!ls -la data/
print("\n✅ 데이터 다운로드 완료!")

## 3. XML → YOLO 형식 변환

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# 출력 폴더 생성
os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

# XML → YOLO 변환 함수
def convert_xml_to_yolo(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)

    yolo_lines = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        center_x = ((xmin + xmax) / 2) / img_width
        center_y = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height

        yolo_lines.append(f"0 {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
    return '\n'.join(yolo_lines)

# XML 파일 찾기
xml_files = []
for root_dir, dirs, files in os.walk('data/'):
    for file in files:
        if file.endswith('.xml'):
            xml_files.append(os.path.join(root_dir, file))

print(f"XML 파일: {len(xml_files)}개")

# Train/Val 분리 (80:20)
train_files, val_files = train_test_split(xml_files, test_size=0.2, random_state=42)
print(f"Train: {len(train_files)}개, Val: {len(val_files)}개")

In [ ]:
# 변환 실행
success_count = 0

for split, files in [('train', train_files), ('val', val_files)]:
    for xml_path in files:
        try:
            xml_name = os.path.splitext(os.path.basename(xml_path))[0]
            
            # 이미지 찾기
            img_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                test_path = f'data/images/{xml_name}{ext}'
                if os.path.exists(test_path):
                    img_path = test_path
                    break

            if img_path:
                img_name = os.path.basename(img_path)
                shutil.copy(img_path, f'dataset/images/{split}/{img_name}')
                
                yolo_label = convert_xml_to_yolo(xml_path)
                with open(f'dataset/labels/{split}/{xml_name}.txt', 'w') as f:
                    f.write(yolo_label)
                success_count += 1
        except Exception as e:
            print(f"Error: {xml_path} - {e}")

print(f"\n✅ 데이터셋 준비 완료! ({success_count}개)")
print(f"Train: {len(os.listdir('dataset/images/train'))}개")
print(f"Val: {len(os.listdir('dataset/images/val'))}개")

## 4. dataset.yaml 생성

In [ ]:
yaml_content = """path: /content/dataset
train: images/train
val: images/val

nc: 1
names: ['plate']
"""

with open('dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ dataset.yaml 생성 완료!")

## 5. YOLO 학습

In [ ]:
from ultralytics import YOLO

# YOLOv8n 모델 로드
model = YOLO('yolov8n.pt')

# 학습 (약 4-5분 소요)
results = model.train(
    data='dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='plate_detector',
    patience=10  # EarlyStopping
)

print("\n✅ YOLO 학습 완료!")

In [ ]:
# 학습 결과 확인
from IPython.display import Image, display

print("=== 학습 곡선 ===")
display(Image(filename='runs/detect/plate_detector/results.png', width=800))

---

# Part B: Step 2+3 - EasyOCR 문자 인식

학습된 YOLO 모델로 번호판 검출 → EasyOCR로 문자 인식

---

## 6. 모델 로드

### 옵션 A: 방금 학습한 모델 사용

In [ ]:
from ultralytics import YOLO
import easyocr

# YOLO 모델 로드 (방금 학습한 모델)
detector = YOLO('runs/detect/plate_detector/weights/best.pt')
print("✅ YOLO 모델 로드 완료!")

# EasyOCR 리더 초기화 (최초 실행 시 모델 다운로드 1-2분)
print("\nEasyOCR 모델 다운로드 중... (최초 1회만)")
reader = easyocr.Reader(['ko', 'en'], gpu=True)
print("✅ EasyOCR 모델 로드 완료!")

### 옵션 B: Google Drive에서 기존 모델 로드 (학습 건너뛰기)

In [ ]:
# # Google Drive 마운트
# from google.colab import drive
# drive.mount('/content/drive')

# from ultralytics import YOLO
# import easyocr

# # Drive에서 모델 로드
# model_path = '/content/drive/MyDrive/AI_Practice/03-LicensePlate/models/plate_detector.pt'
# detector = YOLO(model_path)
# print(f"✅ YOLO 모델 로드: {model_path}")

# # EasyOCR
# reader = easyocr.Reader(['ko', 'en'], gpu=True)
# print("✅ EasyOCR 모델 로드 완료!")

## 7. 번호판 검출 + OCR 통합 함수 정의

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

def detect_and_read_plate(image_path, detector, reader, show=True):
    """
    번호판 검출 + OCR 통합 함수
    
    Args:
        image_path: 이미지 경로
        detector: YOLO 모델
        reader: EasyOCR 리더
        show: 결과 시각화 여부
    
    Returns:
        results: 검출된 번호판 정보 리스트
    """
    # 이미지 로드
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Step 1: 번호판 검출 (YOLO)
    detections = detector(image, verbose=False)
    
    results = []
    result_image = image_rgb.copy()
    
    for box in detections[0].boxes:
        # 바운딩 박스 좌표
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        det_conf = box.conf[0].item()
        
        # 번호판 영역 크롭 (여유 공간 추가)
        margin = 5
        y1_crop = max(0, y1 - margin)
        y2_crop = min(image.shape[0], y2 + margin)
        x1_crop = max(0, x1 - margin)
        x2_crop = min(image.shape[1], x2 + margin)
        
        plate_img = image[y1_crop:y2_crop, x1_crop:x2_crop]
        
        # Step 2+3: OCR 문자 인식 (EasyOCR)
        ocr_results = reader.readtext(plate_img)
        
        # 텍스트 합치기
        if ocr_results:
            plate_text = ' '.join([text for _, text, _ in ocr_results])
            ocr_conf = np.mean([conf for _, _, conf in ocr_results])
        else:
            plate_text = "(인식 실패)"
            ocr_conf = 0.0
        
        results.append({
            'text': plate_text,
            'detection_conf': det_conf,
            'ocr_conf': ocr_conf,
            'bbox': (x1, y1, x2, y2)
        })
        
        # 결과 이미지에 표시
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 255, 0), 3)
    
    # 시각화
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # 원본 이미지 + 바운딩 박스
        axes[0].imshow(result_image)
        axes[0].set_title('Detection Result')
        axes[0].axis('off')
        
        # 번호판 크롭 + OCR 결과
        if results:
            r = results[0]
            x1, y1, x2, y2 = r['bbox']
            plate_crop = image_rgb[y1:y2, x1:x2]
            axes[1].imshow(plate_crop)
            axes[1].set_title(f"OCR: {r['text']}\nConf: {r['ocr_conf']:.1%}")
            axes[1].axis('off')
        else:
            axes[1].text(0.5, 0.5, 'No plate detected', ha='center', va='center')
            axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return results

print("✅ 함수 정의 완료!")

## 8. 테스트 실행

In [ ]:
import os

# Val 이미지로 테스트
val_dir = 'dataset/images/val'
test_images = os.listdir(val_dir)[:5]  # 처음 5개

print(f"테스트 이미지 {len(test_images)}개\n")
print("=" * 60)

all_results = []
for img_name in test_images:
    img_path = os.path.join(val_dir, img_name)
    print(f"\n📷 {img_name}")
    
    results = detect_and_read_plate(img_path, detector, reader)
    
    for i, r in enumerate(results):
        print(f"  [{i+1}] 번호판: {r['text']}")
        print(f"      검출 신뢰도: {r['detection_conf']:.1%}")
        print(f"      OCR 신뢰도: {r['ocr_conf']:.1%}")
    
    if not results:
        print("  번호판을 찾지 못했습니다.")
    
    all_results.extend(results)

print("\n" + "=" * 60)
print(f"\n✅ 총 {len(all_results)}개 번호판 인식 완료!")

## 9. 직접 이미지 업로드하여 테스트

In [ ]:
from google.colab import files

print("테스트할 자동차 이미지를 업로드하세요:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n📷 업로드된 이미지: {filename}")
    print("=" * 50)
    
    results = detect_and_read_plate(filename, detector, reader)
    
    if results:
        for i, r in enumerate(results):
            print(f"\n🚗 번호판 {i+1}")
            print(f"   텍스트: {r['text']}")
            print(f"   검출 신뢰도: {r['detection_conf']:.1%}")
            print(f"   OCR 신뢰도: {r['ocr_conf']:.1%}")
    else:
        print("\n❌ 번호판을 찾지 못했습니다.")

## 10. 결과 요약 및 분석

In [ ]:
import pandas as pd

# 전체 Val 이미지 테스트
print("전체 Val 이미지 테스트 중...")

all_results = []
val_images = os.listdir(val_dir)

for img_name in val_images:
    img_path = os.path.join(val_dir, img_name)
    results = detect_and_read_plate(img_path, detector, reader, show=False)
    
    for r in results:
        all_results.append({
            'image': img_name,
            'plate_text': r['text'],
            'det_conf': r['detection_conf'],
            'ocr_conf': r['ocr_conf']
        })

# DataFrame으로 변환
df = pd.DataFrame(all_results)

print(f"\n=== 결과 요약 ===")
print(f"총 이미지: {len(val_images)}개")
print(f"검출된 번호판: {len(df)}개")

if len(df) > 0:
    print(f"\n평균 검출 신뢰도: {df['det_conf'].mean():.1%}")
    print(f"평균 OCR 신뢰도: {df['ocr_conf'].mean():.1%}")
    
    print(f"\n=== 샘플 결과 (상위 10개) ===")
    display(df.head(10))

## 11. 모델 저장

In [ ]:
# Google Drive에 저장
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/AI_Practice/03-LicensePlate/models'
os.makedirs(save_dir, exist_ok=True)

# YOLO 모델 저장
shutil.copy('runs/detect/plate_detector/weights/best.pt', f'{save_dir}/plate_detector.pt')
print(f"✅ YOLO 모델 저장: {save_dir}/plate_detector.pt")

print("\n💡 EasyOCR 모델은 자동 다운로드되므로 별도 저장 불필요")

In [ ]:
# 또는 직접 다운로드
from google.colab import files
files.download('runs/detect/plate_detector/weights/best.pt')

---

## 완료!

### 파이프라인 요약

```
┌─────────────────────────────────────────────────────────────┐
│  Step 1: YOLO 번호판 검출                                    │
│  ├─ 모델: YOLOv8n (직접 학습)                                │
│  ├─ 데이터: Kaggle 433장                                    │
│  └─ 결과: mAP50 ≈ 88%                                       │
├─────────────────────────────────────────────────────────────┤
│  Step 2+3: EasyOCR 문자 인식                                 │
│  ├─ 모델: 사전 학습 (자동 다운로드)                           │
│  └─ 지원: 한국어 + 영어                                      │
└─────────────────────────────────────────────────────────────┘
```

### 저장된 파일
- `Google Drive/AI_Practice/03-LicensePlate/models/plate_detector.pt`

### 로컬 PC에서 실행
```bash
cd ai-practice/03-LicensePlate/experiments/step1_detection
python detect_and_read.py --image test.jpg
```